# Spotify Data Analysis — Part 1: Data Collection

## System Architecture

The collection pipeline was designed as 5 components (see `architecture/Defining Components.odt`):

| Component | Role |
|-----------|------|
| **Spotify Connector** | Authenticates with Spotify, fetches liked songs & top tracks, writes to DB |
| **Shironet Connector** | Receives song info, scrapes shironet.mako.co.il, returns Hebrew lyrics |
| **Genius Connector** | Receives song info, queries the Genius API, returns English lyrics |
| **Router** | Detects the song language and routes to the correct lyrics connector |
| **Main** | Orchestrates the full pipeline: Spotify → DB → Router → lyrics → DB |

```
User ──► Main ──► Spotify Connector ──► DB
                        │
                  (for each song)
                        │
                     Router
                    /       \
            Shironet         Genius
            (Hebrew)         (English)
```

In [ ]:
import sqlite3
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import io
from IPython.display import SVG, display as _ipy_display

DB_PATH = "spotify_intel.db"

def q(sql):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn)

# SVG rendering — fixes Hebrew BiDi text
matplotlib.rcParams["svg.fonttype"] = "none"
def _svg_show(*args, **kwargs):
    buf = io.BytesIO()
    plt.savefig(buf, format="svg", bbox_inches="tight")
    buf.seek(0)
    _ipy_display(SVG(buf.read()))
    plt.close()
plt.show = _svg_show
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"#f8f9fa",
    "axes.spines.top":False,"axes.spines.right":False,"axes.grid":False,
    "font.size":11,"font.family":"sans-serif",
    "font.sans-serif":["Segoe UI","Tahoma","Arial","DejaVu Sans"]})
C1 = "#6c5ce7"; C2 = "#dfe6e9"
print("Connected to", DB_PATH)

---
## Component 1 — Spotify Connector

**HLD:** Receives user credentials / API key, connects to Spotify, extracts liked songs (name, artist, album, release year, genre) and top tracks for all three time ranges, writes all to the database.

**LLD flow:**
1. Get login details and connect to the Spotify account (OAuth 2.0)
2. Navigate to liked songs
3. For each song: copy title, artist, album, release year, genre
4. Record all data to the database (primary key = Spotify song ID)
5. Repeat for short-term, medium-term and long-term top tracks

In [ ]:
# ── Reference implementation (Spotify Connector) ────────────────────
# class SpotifyClient:
#     async def get_liked_songs(self) -> list[dict]:
#         songs, offset = [], 0
#         while True:
#             batch = self.sp.current_user_saved_tracks(limit=50, offset=offset)
#             if not batch["items"]: break
#             for item in batch["items"]:
#                 t = item["track"]
#                 songs.append({"name": t["name"],
#                               "artist": t["artists"][0]["name"],
#                               "album":  t["album"]["name"],
#                               "release_year": int(t["album"]["release_date"][:4])})
#             offset += 50
#         return songs
#
#     async def get_top_tracks(self, term) -> list[dict]:
#         results = self.sp.current_user_top_tracks(limit=50, time_range=term)
#         return [{"name": t["name"], "term": term, "rank": i+1,
#                  "play_weight": {"short_term":3,"medium_term":2,"long_term":1}[term]}
#                 for i, t in enumerate(results["items"])]

# ── What was actually collected ──────────────────────────────────────
by_term = q("""
    SELECT ls.term, COUNT(*) AS tracks,
           ROUND(AVG(ls.play_weight),1) AS avg_weight
    FROM listening_snapshot ls
    WHERE ls.term IN ("short_term","medium_term","long_term")
    GROUP BY ls.term ORDER BY avg_weight DESC
""")
display(by_term)

liked = q("SELECT COUNT(*) AS liked_songs FROM listening_snapshot WHERE term='liked'")
print(f"Liked songs collected: {liked['liked_songs'][0]}")

In [ ]:
# Sample of collected tracks
sample = q("""
    SELECT t.name, t.artist, t.album, t.release_year
    FROM tracks t
    JOIN listening_snapshot ls ON ls.track_id = t.id
    WHERE ls.term = 'liked'
    ORDER BY RANDOM() LIMIT 8
""")
display(sample)

---
## Component 2 — Router

**HLD:** Receives information about one song and decides which connector to use: **Shironet** (Hebrew) or **Genius** (English).

**LLD flow:**
1. Receive song info (title, artist)
2. Check for Hebrew characters (Unicode range U+05D0–U+05EA) in title or artist name
3. Return the name of the connector: `"shironet"` or `"genius"`

In [ ]:
# ── Reference implementation (Router) ───────────────────────────────
# def is_hebrew(text: str) -> bool:
#     return any("\u05d0" <= ch <= "\u05ea" for ch in (text or ""))
#
# def route(title: str, artist: str) -> str:
#     return "shironet" if is_hebrew(title) or is_hebrew(artist) else "genius"

# ── Simulate routing on all collected tracks ─────────────────────────
def has_hebrew(text):
    if not isinstance(text, str): return False
    return any("\u05d0" <= ch <= "\u05ea" for ch in text)

all_tracks = q("SELECT name, artist FROM tracks")
all_tracks["routed_to"] = all_tracks.apply(
    lambda r: "shironet" if has_hebrew(r["name"]) or has_hebrew(r["artist"]) else "genius",
    axis=1)

routing = all_tracks["routed_to"].value_counts().reset_index()
routing.columns = ["connector", "songs_routed"]
display(routing)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(routing["connector"], routing["songs_routed"], color=[C1, C2])
ax.set_title("Songs Routed by Language", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of songs")
for i, v in enumerate(routing["songs_routed"]):
    ax.text(i, v + 3, str(v), ha="center", fontsize=11)
plt.tight_layout()
plt.show()

---
## Component 3 — Genius Connector (English Lyrics)

**HLD:** Receives song info, connects to the Genius API, returns the lyrics.

**LLD flow:**
1. Receive: song title, artist, album, release year
2. Search the Genius API for the song page URL
3. Scrape the lyrics from the song page
4. Copy lyrics to the DB

In [ ]:
# ── Reference implementation (Genius Connector) ──────────────────────
# class GeniusClient:
#     async def get_lyrics(self, title: str, artist: str) -> str | None:
#         params = {"q": f"{title} {artist}"}
#         r = await client.get(f"{GENIUS_BASE}/search",
#                              headers={"Authorization": f"Bearer {GENIUS_TOKEN}"},
#                              params=params)
#         hits = r.json()["response"]["hits"]
#         if not hits: return None
#         return await self._scrape_page(hits[0]["result"]["url"])

# ── Verify: sample English tracks with lyrics ─────────────────────────
eng_lyrics = q("""
    SELECT t.name, t.artist, length(tl.lyrics_raw) AS chars
    FROM tracks t
    JOIN track_lyrics tl ON tl.track_id = t.id
    WHERE tl.lyrics_raw IS NOT NULL
      AND t.name NOT LIKE '%א%' AND t.name NOT LIKE '%ב%'
      AND t.artist NOT LIKE '%א%' AND t.artist NOT LIKE '%ב%'
    ORDER BY RANDOM() LIMIT 5
""")
display(eng_lyrics)

---
## Component 4 — Shironet Connector (Hebrew Lyrics)

**HLD:** Receives song info, connects to the Shironet website (Israeli lyrics site), returns the Hebrew lyrics.

**LLD flow:**
1. Receive: song title, artist, album, release year
2. Build search URL: `shironet.mako.co.il/search?q=<title>+<artist>`
3. Parse HTML with BeautifulSoup, find the first result link
4. Follow the link to the song page
5. Extract the lyrics block from the page
6. Copy lyrics to the DB

In [ ]:
# ── Reference implementation (Shironet Connector) ────────────────────
# class ShironetClient:
#     BASE = "https://www.shironet.mako.co.il"
#
#     async def get_lyrics(self, title: str, artist: str) -> str | None:
#         url  = f"{self.BASE}/search?q={title}+{artist}"
#         page = await client.get(url, headers=HEADERS)
#         soup = BeautifulSoup(page.text, "html.parser")
#         link = soup.select_one("a.search_result_title_link")
#         if not link: return None
#         song  = await client.get(self.BASE + link["href"])
#         lsoup = BeautifulSoup(song.text, "html.parser")
#         block = lsoup.select_one("p.artist_lyrics_text")
#         return block.get_text("\n") if block else None

# ── Verify: sample Hebrew tracks with lyrics ──────────────────────────
heb_lyrics = q("""
    SELECT t.name, t.artist, length(tl.lyrics_raw) AS chars
    FROM tracks t
    JOIN track_lyrics tl ON tl.track_id = t.id
    WHERE tl.lyrics_raw IS NOT NULL
      AND (t.name LIKE '%א%' OR t.artist LIKE '%א%')
    ORDER BY RANDOM() LIMIT 5
""")
display(heb_lyrics)

---
## Component 5 — Main Pipeline

**HLD:** Invokes the Spotify Connector, gets song data from the DB, sends one song at a time to the Router to get lyrics, stores the lyrics back in the DB.

**LLD flow:**
1. Get login details from the user
2. Contact Spotify Connector → connect & save songs to DB
3. Fetch all songs from DB
4. For each song: Router → get connector name → fetch lyrics
5. Record lyrics in DB, one song at a time
6. Report: total songs, lyrics found, lyrics missing

In [ ]:
# ── Reference implementation (Main pipeline) ─────────────────────────
# async def run_pipeline(user_id, access_token, db, publish=None):
#     # Step 1-2: Spotify Connector
#     async with SpotifyClient(access_token) as sp:
#         liked = await sp.get_liked_songs()
#         top   = await sp.get_top_tracks()
#     await upsert_tracks(liked + top, db)
#
#     # Step 3-5: Router + connectors
#     songs = await db.execute("SELECT id, name, artist FROM tracks")
#     found, failed = 0, 0
#     for song in songs:
#         connector = route(song.name, song.artist)
#         client    = ShironetClient() if connector=="shironet" else GeniusClient()
#         lyrics    = await client.get_lyrics(song.name, song.artist)
#         if lyrics:
#             await save_lyrics(song.id, lyrics, db); found += 1
#         else:
#             failed += 1
#
#     # Step 6: Report
#     return {"total": found+failed, "found": found, "failed": failed}

# ── Pipeline results ──────────────────────────────────────────────────
results = q("""
    SELECT
        (SELECT COUNT(*) FROM tracks)                                        AS total_tracks,
        (SELECT COUNT(*) FROM listening_snapshot WHERE term='liked')         AS liked_songs,
        (SELECT COUNT(*) FROM track_lyrics WHERE lyrics_raw IS NOT NULL)     AS lyrics_found,
        (SELECT COUNT(*) FROM tracks)
            - (SELECT COUNT(*) FROM track_lyrics WHERE lyrics_raw IS NOT NULL) AS lyrics_missing
""")
display(results)

found = int(results["lyrics_found"][0])
total = int(results["total_tracks"][0])
miss  = int(results["lyrics_missing"][0])
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie([found, miss],
       labels=[f"Lyrics found\n{found}", f"Not found\n{miss}"],
       colors=[C1, C2], startangle=90, wedgeprops=dict(width=0.5),
       textprops=dict(fontsize=11))
ax.set_title("Pipeline Result — Lyrics Coverage", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()
print(f"\nPipeline success rate: {round(found/total*100,1)}%")